# Aufgabe

Macht es Sinn die Spalte "fuelType" und "gearBox" mit in das Modell einfließen zu lassen? Können wir damit unser Modell verbessern (Bestimmtheitsmaß)?


In [34]:
import pandas as pd

df = pd.read_csv("./data/Autos/autos.csv.bz2", encoding="ISO-8859-1")

df = df[df["offerType"] == "Angebot"]
df = df[df["vehicleType"] == "kleinwagen"]
df = df[df["notRepairedDamage"] == "nein"]
df = df[(df["fuelType"] == "benzin") | (df["fuelType"] == "diesel") | (df["fuelType"] == "hybrid")]

df.dropna(inplace=True)

df.head()

,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
3,2016-03-17 16:54:04,GOLF_4_1_4__3TÜRER,privat,Angebot,1500,test,kleinwagen,2001,manuell,75,golf,150000,6,benzin,volkswagen,nein,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,Skoda_Fabia_1.4_TDI_PD_Classic,privat,Angebot,3600,test,kleinwagen,2008,manuell,69,fabia,90000,7,diesel,skoda,nein,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21
17,2016-03-20 10:25:19,Renault_Twingo_1.2_16V_Aut.,privat,Angebot,1750,control,kleinwagen,2004,automatik,75,twingo,150000,2,benzin,renault,nein,2016-03-20 00:00:00,0,65599,2016-04-06 13:16:07
23,2016-03-12 19:43:07,Stadtflitzer,privat,Angebot,450,test,kleinwagen,1997,manuell,50,arosa,150000,5,benzin,seat,nein,2016-03-12 00:00:00,0,9526,2016-03-21 01:46:11
29,2016-03-08 19:55:19,Fiat_Punto_1.2,privat,Angebot,690,test,kleinwagen,2003,manuell,60,punto,150000,3,benzin,fiat,nein,2016-03-08 00:00:00,0,86199,2016-03-09 11:45:28


In [35]:
df["fuelType"].unique()

array(['benzin', 'diesel', 'hybrid'], dtype=object)

## Model Score Basis

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df[["yearOfRegistration", "kilometer"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))


0.5612320996566951
0.5497759999700532


## Model Score mit Gearbox und fuelType im Modell 

### Nur Gearbox

In [37]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

X = df[["yearOfRegistration", "kilometer", "brand", "gearbox"]]
ct = ColumnTransformer(
    [("gearbox", OneHotEncoder(drop = "first"), ["gearbox"]),
     ("brand", OneHotEncoder(drop = "first"), ["brand"])],
    remainder="passthrough"
)
X.head()

ct.fit(X)
X_transformed = ct.transform(X)

In [38]:
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

0.6605552346598177
0.6594261185918693


### Nur fuelType

In [39]:
X = df[["yearOfRegistration", "kilometer", "brand", "fuelType"]]
ct = ColumnTransformer(
    [("fuelType", OneHotEncoder(drop = "first"), ["fuelType"]),
     ("brand", OneHotEncoder(drop = "first"), ["brand"])],
    remainder="passthrough"
)
X.head()

ct.fit(X)
X_transformed = ct.transform(X)

In [40]:
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

0.6304206008251247
0.6674068491171886


### fuelType + gearBox

In [41]:
X = df[["yearOfRegistration", "kilometer", "brand", "fuelType", "gearbox"]]
ct = ColumnTransformer(
    [("fuelType", OneHotEncoder(drop = "first"), ["fuelType"]),
     ("brand", OneHotEncoder(drop = "first"), ["brand"]),
     ("gearbox", OneHotEncoder(drop = "first"), ["gearbox"])],
    remainder="passthrough"
)
X.head()

ct.fit(X)
X_transformed = ct.transform(X)

In [43]:
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
print(model.score(X_train, y_train))

0.6645976886284116
0.6618953876583025


In [47]:
scores = []

for i in range(0, 1000):
    X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2)

    model = LinearRegression()
    model.fit(X_train, y_train)

    scores.append(model.score(X_test, y_test))
    
scores.sort(reverse=True)
print(scores)

[0.7031841949757055, 0.6979722628834907, 0.6959692207689013, 0.6953716276712025, 0.6944981548255528, 0.6942150684482287, 0.6937841636098273, 0.6935560747266868, 0.6935060005701015, 0.6931479975509558, 0.6929770886453039, 0.6926343164209923, 0.6925411230263063, 0.6921978553131944, 0.6916544612892896, 0.6913085971164239, 0.6911192677552866, 0.6909501213592963, 0.6906026218884717, 0.6905836917270181, 0.6904842258355168, 0.6904287056831104, 0.6903497204944912, 0.6901990282686614, 0.6901747563100129, 0.6899973766649505, 0.6899075056565207, 0.6898642589320774, 0.6895291421643911, 0.6894760259368122, 0.6893746493799302, 0.6893017905790173, 0.6893015374129867, 0.6892736994646227, 0.6892028150907983, 0.6891385537441526, 0.6890777978313207, 0.6890623808800629, 0.6890347657487788, 0.688994378464924, 0.6888180319620673, 0.6887498606753613, 0.6887167685465905, 0.6886529180418418, 0.6886161748460227, 0.6885792683856315, 0.6885468835931905, 0.6885106026292616, 0.6880151776908445, 0.6879902613642835, 

In [48]:
import numpy as np

print(np.mean(scores))

0.6618307775583573
